In [2]:
import os
import uuid
from langgraph.checkpoint.memory import MemorySaver
from langgraph.constants import START,END
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langgraph.types import interrupt, Command
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from typing import Annotated

In [ ]:

 

def add_list(a,b):
    return a+b
class State(TypedDict):
    a:Annotated[list[str],add_list]
    user_name:str
def node1(state:State):
    return {"a":["hello1"]}
def node2(state:State):
    user_name=interrupt({"hello":"world"})
    return {"a":["bye"],"user_name":user_name}


def node3(state:State):
    return {"a":["sk"]}
def node4(state:State):
    return {"a":["yo"]}
def node5(state:State):
    return {"a":["haah"]}
builder=StateGraph(State)
builder.add_node("node1",node1)
builder.add_node("node2",node2)
builder.add_node("node3",node3)
builder.add_node("node4",node4)
builder.add_node("node5",node5)
builder.add_edge(START,"node1")
builder.add_edge("node1","node2")
builder.add_edge("node2","node3")
builder.add_edge("node3","node4")
builder.add_edge("node4","node5")
builder.add_edge("node5",END)
builder.add_node("node69",node1)




In [ ]:
from langgraph.checkpoint.memory import MemorySaver

checkpointer = MemorySaver()

"state will be preserved over different llm calls for different thread"

In [19]:
graph = builder.compile(
    checkpointer=checkpointer # Required for `interrupt` to work
)

In [92]:
thread_config = {"configurable": {"thread_id": "5"}}
graph.invoke({"a":[],"user_name":""},config=thread_config)

{'a': ['hello1',
  'bye',
  'sk',
  'yo',
  'haah',
  'hello1',
  'bye',
  'sk',
  'yo',
  'haah',
  'hello1'],
 'user_name': ''}

In [88]:
graph.invoke(Command(resume="shashank"),config=thread_config)

{'a': ['hello1',
  'bye',
  'sk',
  'yo',
  'haah',
  'hello1',
  'bye',
  'sk',
  'yo',
  'haah'],
 'user_name': 'shashank'}

In [73]:
config = {"configurable": {"thread_id": "5"}}
for i in graph.get_state(config):
    print(i)


{'a': ['hello1', 'bye', 'sk', 'yo', 'haah', 'hello1'], 'user_name': ''}
('node2',)
{'configurable': {'thread_id': '5', 'checkpoint_ns': '', 'checkpoint_id': '1f00d08c-f774-640e-8008-133567d2e15c'}}
{'source': 'loop', 'writes': {'node1': {'a': ['hello1']}}, 'thread_id': '5', 'step': 8, 'parents': {}}
2025-03-30T01:46:34.008474+00:00
{'configurable': {'thread_id': '5', 'checkpoint_ns': '', 'checkpoint_id': '1f00d08c-f774-640d-8007-75719dd9d9b8'}}
(PregelTask(id='c8d408a3-4265-8e65-225d-f8d6f5301b76', name='node2', path=('__pregel_pull', 'node2'), error=None, interrupts=(Interrupt(value={'hello': 'world'}, resumable=True, ns=['node2:c8d408a3-4265-8e65-225d-f8d6f5301b76']),), state=None, result=None),)


In [ ]:
config = {"configurable": {"thread_id": "2","checkpoint_id":"1f00cf55-c532-6e9d-8005-81ca3f41bb32"}}
graph.get_state(config)


1) Pass a value to the interrupt: Provide data, such as a user's response, to the graph using Command(resume=value). Execution resumes from the beginning of the node where the interrupt was used, however, this time the interrupt(...) call will return the value passed in the Command(resume=value) instead of pausing the graph.



In [ ]:
 
# Resume graph execution with the user's input.
graph.invoke(Command(resume={"age": "25"}), thread_config)
 

2) Update the graph state: Modify the graph state using Command(update=update). Note that resumption starts from the beginning of the node where the interrupt was used. Execution resumes from the beginning of the node where the interrupt was used, but with the updated state.



In [ ]:
# Update the graph state and resume.
# You must provide a `resume` value if using an `interrupt`.

graph.invoke(Command(update={"foo": "bar"}, resume="Let's go!!!"), thread_config)

In [93]:
config = {"configurable": {"thread_id": "5"}}
state=graph.get_state(config)

In [94]:
state.values  #---->give current state values

{'a': ['hello1',
  'bye',
  'sk',
  'yo',
  'haah',
  'hello1',
  'bye',
  'sk',
  'yo',
  'haah',
  'hello1'],
 'user_name': ''}

In [ ]:
state.tasks  #----> give info about interrupts

1

In [99]:
for i in state.tasks[0]:
    print(i)

04032311-4c4d-2f57-00fd-e646bbfa5460
node2
('__pregel_pull', 'node2')
None
(Interrupt(value={'hello': 'world'}, resumable=True, ns=['node2:04032311-4c4d-2f57-00fd-e646bbfa5460']),)
None
None


Place code with side effects, such as API calls, after the interrupt to avoid duplication, as these are re-triggered every time the node is resumed.



## To update State manually

In [ ]:
graph.update_state(config, {"input": "hello universe!"})